# The Fourier ratio and signal recovery
Two real signals have equal energy, identical Fourier phases, and all 64 Fourier coefficients nonzero. One magnifies two pairs of frequencies. The other has a flat spectrum. Both are sampled at the same times.

Run the cells in order in a Python 3 notebook with NumPy and Matplotlib. The experiment below used NumPy 2.3.5 and Matplotlib 3.10.8. The final cell writes the plot to the current directory.

In [1]:
import numpy as np

n = 64
rng = np.random.default_rng(2026)
keep = rng.random(n) < 0.5
phase = rng.uniform(0, 2 * np.pi, n // 2 - 1)

def signal(amplitudes):
    positive = amplitudes[1:-1] * np.exp(1j * phase)
    spectrum = np.r_[amplitudes[0], positive, amplitudes[-1], positive[::-1].conj()]
    values = np.fft.ifft(spectrum, norm="ortho").real
    return values / np.linalg.norm(values)

concentrated = np.full(n // 2 + 1, 0.005)
concentrated[[3, 7]] = [1.0, 0.55]
signals = {
    "Two dominant tones": signal(concentrated),
    "Equal amplitudes": signal(np.ones(n // 2 + 1)),
}

## Recovery from observed values
We minimize the sum of the magnitudes of the complex Fourier coefficients, subject to agreement with the observed values. The alternating direction method of multipliers (ADMM) alternates projection onto those constraints with complex soft thresholding. The solver uses only the sample values and their locations. We stop when the absolute primal and dual residuals are both below $10^{-7}$.

In [2]:
def recover(observed, keep, rho=10.0, tolerance=1e-7):
    # ADMM for min ||c||_1 subject to inverse_DFT(c)[keep] = observed.
    # Projection is exact since the DFT is unitary.
    z = np.zeros(keep.size, dtype=complex)
    dual = np.zeros_like(z)
    for iteration in range(100000):
        values = np.fft.ifft(z - dual, norm="ortho")
        values[keep] = observed
        c = np.fft.fft(values, norm="ortho")
        argument = c + dual
        magnitude = np.maximum(np.abs(argument), np.finfo(float).tiny)
        previous = z
        z = argument * np.maximum(0, 1 - 1 / (rho * magnitude))
        dual += c - z
        primal_error = np.linalg.norm(c - z)
        dual_error = rho * np.linalg.norm(z - previous)
        if max(primal_error, dual_error) < tolerance:
            assert np.max(np.abs(values.imag)) < 1e-10
            return values.real
    raise RuntimeError("The recovery solver did not converge.")

## Compare the two signals
Compute the Fourier ratio $\|\widehat f\|_1/\|\widehat f\|_2$ and the relative error $\|f-g\|_2/\|f\|_2$. The true signal is used to score reconstruction after fitting.

In [3]:
results = {}
print(f"Observed {keep.sum()} of {n} times")
for name, truth in signals.items():
    spectrum = np.abs(np.fft.fft(truth, norm="ortho"))
    estimate = recover(truth[keep], keep)
    ratio = spectrum.sum() / np.linalg.norm(spectrum)
    error = np.linalg.norm(estimate - truth) / np.linalg.norm(truth)
    assert np.allclose(estimate[keep], truth[keep], atol=1e-10)
    results[name] = spectrum, estimate, ratio, error
    print(f"{name}: Fourier ratio {ratio:.3f}, relative error {100 * error:.2f}%")

Observed 35 of 64 times
Two dominant tones: Fourier ratio 2.106, relative error 3.14%
Equal amplitudes: Fourier ratio 8.000, relative error 88.80%


In [4]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({"svg.fonttype": "none", "svg.hashsalt": "fourier-ratio-2026",
                     "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
fig, axes = plt.subplots(2, 2, figsize=(8, 5.4), width_ratios=[1, 2], layout="constrained")
times = np.arange(n)
for row, (name, truth) in enumerate(signals.items()):
    spectrum, estimate, ratio, error = results[name]
    left, right = axes[row]
    left.vlines(np.arange(n // 2 + 1), 0, spectrum[:n // 2 + 1],
                color="#275774", linewidth=1.8)
    left.set(xlabel="Frequency", ylabel="Magnitude", ylim=(0, 0.67),
             xticks=[0, 8, 16, 24, 32])
    left.set_title(name, loc="left", fontweight="bold", fontsize=12)
    right.plot(times, truth, color="#20384a", label="Original", linewidth=1.6)
    right.plot(times, estimate, color="#ad461c", linestyle="--", label="Recovered", linewidth=1.6)
    right.scatter(times[keep], truth[keep], s=12, color="#20384a", zorder=3, label="Observed")
    right.set(xlabel="Time index", ylabel="Signal value", ylim=(-0.42, 0.42),
              xticks=[0, 16, 32, 48, 63], xlim=(-1, n))
    right.set_title(f"Relative error {100 * error:.2f}%", loc="left", fontsize=12)
    for ax in (left, right):
        ax.grid(axis="y", alpha=0.16)
        ax.tick_params(labelsize=9)
axes[0, 1].legend(loc="upper center", bbox_to_anchor=(0.5, 1), ncol=3,
                  frameon=False, fontsize=8, columnspacing=1)
fig.savefig("fourier-ratio-recovery.svg", metadata={"Date": None})
plt.show()

The random sampling theorem is stated as Theorem 3 in [Burstein, Iosevich, and Nathan](https://arxiv.org/abs/2601.16345). The experiment uses exact constraints and a single random sample. Changing the seed changes the sample, phases, and errors.